In [2]:
##### Compiles csv of training data for regions in dataset

import os
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from functools import reduce

In [31]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent 

# folder of predictors 
predictors = Path(f'{cd}/Data/Clean/Predictors/Vectors')

# intensities 
capital = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_capital_intensity.csv")
labor = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_labor_intensity.csv")

# save path
save_path_capital = f'{cd}/Data/Clean/Training_data/capital.csv'
save_path_labor = f'{cd}/Data/Clean/Training_data/labor.csv'

In [29]:
##### Merge all predictors into one df (dropping all duplicate PROJ_ID entries first)
dfs = []
for f in os.listdir(predictors):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join(predictors, f))
        dupes = df['PROJ_ID'].duplicated().sum()
        if dupes > 0:
            df = df.drop_duplicates(subset='PROJ_ID')
        dfs.append(df)

predictors_df = reduce(lambda left, right: pd.merge(left, right, on='PROJ_ID', how='outer'), dfs)

In [30]:
### Merge with capital and labor 

### Capital
capital = capital.drop_duplicates(subset='PROJ_ID')
capital_merge = capital.merge(predictors_df, on='PROJ_ID', how='left')
capital_merge.to_csv(save_path_capital, index=False)

### Labor
labor = labor.drop_duplicates(subset='PROJ_ID')
labor_merge = labor.merge(predictors_df, on='PROJ_ID', how='left')
labor_merge.to_csv(save_path_labor, index=False)